# Step 3 — TOD Station Assignment Re-integration

Reads the manually-reviewed Excel workbook (path set via `REVIEW_XLSX` in `config.py`) produced by
Step 2, validates any manually-assigned `station_id` values against the TOD
station universe, applies the corrections to the development datasets, and
exports the final authoritative `tod_stations`, `tod_stops`, and
`tod_access_points` layers to the shared GeoPackage.

> **Pipeline order:** Run after completing the manual Excel review that follows
> `2_tod_stop_and_access_assignment.ipynb`.
> - Rename `SB79_tod_review.xlsx` to `SB79_tod_review_reviewed_YYYY_MM_DD.xlsx` and update
>   `REVIEW_XLSX` in `config.py` to point to the renamed file.
> - For any conflict or no-match row that needs a corrected station, set
>   `assignment_status` = `manual_station_assignment` and enter the correct
>   `station_id`.
> - Save the workbook, then run this notebook.
>   If any manually-assigned `station_id` values are not found in the TOD station
>   universe, the notebook will raise an error — correct them in the workbook and re-run.
>
> All data source paths and layer names are defined in `config.py`.


In [1]:
import pandas as pd
import geopandas as gpd
from config import *

In [2]:
# All paths and layer names are imported from config.py
tod_gpkg_path = TOD_DATABASE_GPKG

In [3]:
# Load the manually-reviewed Excel workbook (written by Step 2).
# All records are present in each sheet; only rows with
# assignment_status == 'manual_station_assignment' will be applied as updates.
stops_review_df = pd.read_excel(REVIEW_XLSX, sheet_name=REVIEW_STOPS_SHEET, dtype=str)
access_review_df = pd.read_excel(REVIEW_XLSX, sheet_name=REVIEW_ACCESS_PTS_SHEET, dtype=str)

print(f"Loaded review workbook: {REVIEW_XLSX}")
print(f"  Sheet '{REVIEW_STOPS_SHEET}':       {len(stops_review_df)} rows")
print(f"  Sheet '{REVIEW_ACCESS_PTS_SHEET}': {len(access_review_df)} rows")

# Filter to only manually-assigned rows
MANUAL_STATUS = "manual_station_assignment"
stops_manual = stops_review_df[stops_review_df["assignment_status"] == MANUAL_STATUS].copy()
access_manual = access_review_df[access_review_df["assignment_status"] == MANUAL_STATUS].copy()

print(f"\nRows flagged '{MANUAL_STATUS}':")
print(f"  Stops:         {len(stops_manual)}")
print(f"  Access points: {len(access_manual)}")

Loaded review workbook: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_review_reviewed_2026_03_06.xlsx
  Sheet 'stops':       755 rows
  Sheet 'access_points': 1183 rows

Rows flagged 'manual_station_assignment':
  Stops:         22
  Access points: 70


In [4]:
# Load the development datasets written by Step 2 and the TOD station universe
# (used to validate any manually-entered station_id values).
tod_stations_dev = gpd.read_file(tod_gpkg_path, layer=GPKG_TOD_STATIONS_DEV_LAYER)
stops_dev = gpd.read_file(tod_gpkg_path, layer=GPKG_TOD_STOPS_DEV_LAYER)
access_pts_dev = gpd.read_file(tod_gpkg_path, layer=GPKG_TOD_ACCESS_PTS_DEV_LAYER)

print(f"Loaded {len(tod_stations_dev)} TOD stations (dev) from GeoPackage")
print(f"Loaded {len(stops_dev)} TOD stops (dev) from GeoPackage")
print(f"Loaded {len(access_pts_dev)} access points (dev) from GeoPackage")

# Validate: ensure every manually-assigned station_id exists in the TOD station universe
valid_station_ids = set(tod_stations_dev["station_id"].dropna().astype(str))

def _validate_station_ids(manual_df, id_col):
    """Return (valid_df, invalid_df) split on whether station_id is in valid_station_ids."""
    if len(manual_df) == 0:
        return manual_df.copy(), manual_df.iloc[0:0].copy()
    manual_df = manual_df.copy()
    manual_df["station_id"] = manual_df["station_id"].astype(str).str.strip()
    is_valid = manual_df["station_id"].isin(valid_station_ids)
    return manual_df[is_valid], manual_df[~is_valid]

stops_valid, stops_invalid = _validate_station_ids(stops_manual, "stop_id")
access_valid, access_invalid = _validate_station_ids(access_manual, "access_id")

# Raise an error if any manually-assigned station_ids are not in the TOD station universe.
# Correct the station_id values in the review workbook and re-run this cell.
errors = []
if len(stops_invalid) > 0:
    rows = stops_invalid[["stop_id", "station_id"]].to_string(index=False)
    errors.append(f"{len(stops_invalid)} stop row(s) have invalid station_id values:\n{rows}")
if len(access_invalid) > 0:
    rows = access_invalid[["access_id", "station_id"]].to_string(index=False)
    errors.append(f"{len(access_invalid)} access point row(s) have invalid station_id values:\n{rows}")

if errors:
    raise ValueError(
        "Invalid station_id assignments detected — fix these in the review workbook "
        f"({REVIEW_XLSX.name}) and re-run this cell:\n\n" + "\n\n".join(errors)
    )

print(f"\nAll manually-assigned station_ids are valid.")
print(f"\nValid manual assignments to apply:")
print(f"  Stops:         {len(stops_valid)}")
print(f"  Access points: {len(access_valid)}")


Loaded 459 TOD stations (dev) from GeoPackage
Loaded 760 TOD stops (dev) from GeoPackage
Loaded 1183 access points (dev) from GeoPackage


ValueError: Invalid station_id assignments detected — fix these in the review workbook (SB79_tod_review_reviewed_2026_03_06.xlsx) and re-run this cell:

2 access point row(s) have invalid station_id values:
               access_id station_id
geom:37.33636,-121.89165     delete
geom:37.33650,-121.89019     delete

In [ ]:
# Apply validated manual station assignments to the development datasets.
stops_final = stops_dev.copy()
access_pts_final = access_pts_dev.copy()

if len(stops_valid) > 0:
    stops_final = stops_final.set_index("stop_id")
    updates = stops_valid.set_index("stop_id")["station_id"]
    stops_final.loc[stops_final.index.isin(updates.index), "station_id"] = updates
    stops_final = stops_final.reset_index()

if len(access_valid) > 0:
    access_pts_final = access_pts_final.set_index("access_id")
    updates = access_valid.set_index("access_id")["station_id"]
    access_pts_final.loc[access_pts_final.index.isin(updates.index), "station_id"] = updates
    access_pts_final = access_pts_final.reset_index()

# Summary
print("=== POST-REVIEW ASSIGNMENT SUMMARY ===")
print(f"\nTOD stops ({len(stops_final)} total):")
print(f"  Assigned:   {stops_final['station_id'].notna().sum()}")
print(f"  Unassigned: {stops_final['station_id'].isna().sum()}")
print(f"\nAccess points ({len(access_pts_final)} total):")
print(f"  Assigned:   {access_pts_final['station_id'].notna().sum()}")
print(f"  Unassigned: {access_pts_final['station_id'].isna().sum()}")

# All access points must have a station_id before export.
# If any remain unassigned, go back to the review workbook, set
# assignment_status = 'manual_station_assignment', enter the correct
# station_id, save, and re-run from cell 4.
unassigned_access = access_pts_final[access_pts_final["station_id"].isna()]
if len(unassigned_access) > 0:
    rows = unassigned_access[["access_id", "access_point_name", "assignment_status"]].to_string(index=False)
    raise ValueError(
        f"{len(unassigned_access)} access point(s) have no station_id after review — "
        f"assign each one in {REVIEW_XLSX.name} and re-run from cell 4:\n\n{rows}"
    )

print("\nAll access points have a station_id — ready to export.")

In [ ]:
access_pts_final.query("station_id.isna()")[["access_id", "access_point_name"]]

In [ ]:
# Export final authoritative layers to the shared GeoPackage
print(f"\nExporting final layers to GeoPackage: {tod_gpkg_path}")

tod_stations_dev.to_file(tod_gpkg_path, layer=GPKG_TOD_STATIONS_FINAL_LAYER, driver="GPKG", mode="w")
print(f"Exported {len(tod_stations_dev)} stations → '{GPKG_TOD_STATIONS_FINAL_LAYER}'")

stops_final.to_file(tod_gpkg_path, layer=GPKG_TOD_STOPS_FINAL_LAYER, driver="GPKG", mode="w")
print(f"Exported {len(stops_final)} TOD stops → '{GPKG_TOD_STOPS_FINAL_LAYER}'")

access_pts_final.to_file(tod_gpkg_path, layer=GPKG_TOD_ACCESS_PTS_FINAL_LAYER, driver="GPKG", mode="w")
print(f"Exported {len(access_pts_final)} access points → '{GPKG_TOD_ACCESS_PTS_FINAL_LAYER}'")

print(f"\nFinal output: {tod_gpkg_path}")